# YOLO Object Detection Exploration

This notebook walks through the YOLO object detection pipeline using Ultralytics YOLOv8 and OpenCV. It includes model inspection, preprocessing visualization, single-image detection, threshold experiments, IoU visualization, speed benchmarking, and class distribution analysis.

In [ ]:
# Setup imports and environment
import sys
from pathlib import Path

# Ensure the source directory is importable from the notebook.
sys.path.append(str(Path("../src").resolve()))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from preprocessor import ImagePreprocessor
from detector import YOLODetector
from visualizer import DetectionVisualizer

plt.rcParams["figure.figsize"] = (12, 8)

image_path = Path("../input/sample.jpg")
if not image_path.exists():
    raise FileNotFoundError(
        f"Place a sample image into {image_path.parent.resolve()} and name it sample.jpg for this notebook."
    )

print(f"Using sample image: {image_path}")


## Section 1: Load and Inspect the Model

This section loads the YOLOv8 model and prints its basic properties. We inspect the architecture, number of parameters, and supported class names.


In [ ]:
detector = YOLODetector()
detector.get_model_info()

## Section 2: Preprocessing Deep Dive

This section shows the image at each preprocessing step: original image, letterboxed resize, and blob channels. Inspecting these steps helps understand how YOLO preserves object proportions and creates network inputs.


In [ ]:
preprocessor = ImagePreprocessor()
original_image = cv2.imread(str(image_path))
processed_image, metadata = preprocessor.preprocess(original_image)
blob = preprocessor.to_blob(processed_image)

print("Original image shape:", original_image.shape)
print("Processed image shape:", processed_image.shape)
print("Metadata:", metadata)
print("Blob shape:", blob.shape)

fig, axes = plt.subplots(1, 2)
axes[0].imshow(cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor((processed_image * 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
axes[1].set_title("Letterboxed + Normalized Image")
axes[1].axis("off")

plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx in range(3):
    axes[idx].imshow(blob[0, idx, :, :], cmap="gray")
    axes[idx].set_title(f"Blob channel {idx}")
    axes[idx].axis("off")

plt.show()


## Section 3: Single Image Detection

Run inference on a single image, display raw output tensor shapes, and inspect what the network returns before post-processing.


In [ ]:
# Run a single image detection pass
raw_results = detector.model.predict(source=str(image_path), imgsz=detector.input_size, conf=0.0, iou=0.0, device=detector.device, verbose=False)

print("Raw results count:", len(raw_results))
result = raw_results[0]
print("Boxes present:", hasattr(result, 'boxes') and result.boxes is not None)
print("Raw boxes shape:", result.boxes.xyxy.shape if hasattr(result.boxes, 'xyxy') else None)
print("Raw conf shape:", result.boxes.conf.shape if hasattr(result.boxes, 'conf') else None)
print("Raw class shape:", result.boxes.cls.shape if hasattr(result.boxes, 'cls') else None)

# Convert raw predictions into Detection objects without post-processing
raw_detections = []
for bbox, conf, cls in zip(result.boxes.xyxy.cpu().numpy(), result.boxes.conf.cpu().numpy(), result.boxes.cls.cpu().numpy().astype(int)):
    raw_detections.append((bbox, float(conf), int(cls), detector.class_names.get(int(cls), str(int(cls)))))

print(f"First 10 raw detections (bbox, confidence, class_id, class_name):")
for det in raw_detections[:10]:
    print(det)

# Run the detector wrapper for cleaned detections
detections = detector.detect(str(image_path))
print(f"Clean detections count: {len(detections)}")
for det in detections[:10]:
    print(det)


## Section 4: Confidence Threshold Experiment

Explore how different confidence thresholds affect detection results. Lower thresholds keep more boxes but increase false positives; higher thresholds reduce noise but may miss objects.


In [ ]:
thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]
fig, axes = plt.subplots(1, len(thresholds), figsize=(20, 6))

for ax, threshold in zip(axes, thresholds):
    detections = detector.detect(str(image_path)) if threshold == detector.confidence else detector.detect(str(image_path))
    # Rebuild detector with specific threshold for proper results
    temp_detector = YOLODetector(confidence=threshold, nms_threshold=detector.nms_threshold, device=detector.device)
    detections = temp_detector.detect(str(image_path))
    annotated = DetectionVisualizer().draw_detections(original_image, detections)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f"conf={threshold:.1f}\ncount={len(detections)}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## Section 5: NMS Threshold Experiment

Try different NMS IoU thresholds and observe how overlapping boxes are merged or preserved.


In [ ]:
nms_values = [0.2, 0.45, 0.7]
fig, axes = plt.subplots(1, len(nms_values), figsize=(18, 6))

for ax, nms_value in zip(axes, nms_values):
    temp_detector = YOLODetector(confidence=detector.confidence, nms_threshold=nms_value, device=detector.device)
    detections = temp_detector.detect(str(image_path))
    annotated = DetectionVisualizer().draw_detections(original_image, detections)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f"NMS={nms_value:.2f}\ncount={len(detections)}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## Section 6: IoU Visualization

Pick two overlapping detections, draw both boxes, and shade the intersection and union areas. Display the IoU value to make the concept concrete.


In [ ]:
import matplotlib.patches as patches

# Use detector with the current default thresholds for detection
current_detections = detector.detect(str(image_path))

if len(current_detections) < 2:
    raise ValueError("Need at least two detections for IoU visualization.")

# Choose two overlapping detections if available
first, second = current_detections[0], None
for candidate in current_detections[1:]:
    # simple overlap test
    x1 = max(first.bbox[0], candidate.bbox[0])
    y1 = max(first.bbox[1], candidate.bbox[1])
    x2 = min(first.bbox[2], candidate.bbox[2])
    y2 = min(first.bbox[3], candidate.bbox[3])
    if x2 > x1 and y2 > y1:
        second = candidate
        break

if second is None:
    second = current_detections[1]

# Compute IoU

def compute_iou(box_a, box_b):
    x1_a, y1_a, x2_a, y2_a = box_a
    x1_b, y1_b, x2_b, y2_b = box_b
    inter_x1 = max(x1_a, x1_b)
    inter_y1 = max(y1_a, y1_b)
    inter_x2 = min(x2_a, x2_b)
    inter_y2 = min(y2_a, y2_b)
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    area_a = max(0, x2_a - x1_a) * max(0, y2_a - y1_a)
    area_b = max(0, x2_b - x1_b) * max(0, y2_b - y1_b)
    union_area = area_a + area_b - inter_area
    return inter_area / union_area if union_area > 0 else 0.0

ious = compute_iou(first.bbox, second.bbox)
print(f"Selected boxes: {first.class_name} and {second.class_name}")
print(f"IoU = {ious:.4f}")

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB))

rect1 = patches.Rectangle(
    (first.bbox[0], first.bbox[1]), first.bbox[2] - first.bbox[0], first.bbox[3] - first.bbox[1],
    linewidth=2, edgecolor='red', facecolor='none'
)
rect2 = patches.Rectangle(
    (second.bbox[0], second.bbox[1]), second.bbox[2] - second.bbox[0], second.bbox[3] - second.bbox[1],
    linewidth=2, edgecolor='blue', facecolor='none'
)
ax.add_patch(rect1)
ax.add_patch(rect2)

inter_x1 = max(first.bbox[0], second.bbox[0])
inter_y1 = max(first.bbox[1], second.bbox[1])
inter_x2 = min(first.bbox[2], second.bbox[2])
inter_y2 = min(first.bbox[3], second.bbox[3])
if inter_x2 > inter_x1 and inter_y2 > inter_y1:
    inter_rect = patches.Rectangle(
        (inter_x1, inter_y1), inter_x2 - inter_x1, inter_y2 - inter_y1,
        linewidth=0, facecolor='yellow', alpha=0.4
    )
    ax.add_patch(inter_rect)

ax.text(10, 30, f"IoU = {ious:.4f}", color='white', fontsize=14, bbox=dict(facecolor='black', alpha=0.5))
ax.axis('off')
plt.show()

## Section 7: Speed Benchmarking

Time a number of inference runs to understand how long YOLO takes and to inspect latency distribution.


In [ ]:
run_count = 100
latencies = []
for _ in range(run_count):
    start = time.time()
    _ = detector.model.predict(source=str(image_path), imgsz=detector.input_size, conf=detector.confidence, iou=detector.nms_threshold, device=detector.device, verbose=False)
    latencies.append((time.time() - start) * 1000.0)

latencies = np.array(latencies)
print(f"Mean latency: {latencies.mean():.2f} ms")
print(f"Median latency: {np.median(latencies):.2f} ms")
print(f"P95 latency: {np.percentile(latencies, 95):.2f} ms")

plt.hist(latencies, bins=15, color='blue', alpha=0.7)
plt.title('Inference Latency Distribution')
plt.xlabel('Latency (ms)')
plt.ylabel('Count')
plt.show()

## Section 8: Class Distribution Analysis

Run detection on multiple sample images and count detected classes. This shows which objects are most common in a small dataset.


In [ ]:
from collections import Counter
from glob import glob

image_files = sorted(glob(str(Path("../input") / "*.jpg"))) + sorted(glob(str(Path("../input") / "*.png")))
if not image_files:
    raise FileNotFoundError("No sample images found in ../input. Add a few images to the input folder.")

class_counter = Counter()
max_images = min(10, len(image_files))
for image_file in image_files[:max_images]:
    detections = detector.detect(image_file)
    class_counter.update([det.class_name for det in detections])
    print(f"{Path(image_file).name}: {len(detections)} detections")

print("\nAggregated class counts:")
for class_name, count in class_counter.most_common(20):
    print(f"{class_name}: {count}")

labels, values = zip(*class_counter.most_common(15))
plt.figure(figsize=(12, 6))
plt.bar(labels, values, color='green', alpha=0.8)
plt.xticks(rotation=45, ha='right')
plt.title('Top Detected Classes')
plt.ylabel('Count')
plt.tight_layout()
plt.show()